# 1. How Principal Flow Axis Detection Works

principal_axis_bearing(Eas, Nor) function uses Principal Component Analysis (PCA) to find the direction of maximum velocity variance. 

### Step 1: Create Velocity Matrix:
U = np.vstack([Eas, Nor]).T

* Combines East and North velocities into a 2D matrix
* Each row = one time point [East_velocity, North_velocity]
* Shape: [n_timepoints, 2]

### Step 2: Center the Data
U = U - U.mean(axis=0, keepdims=True)

* Removes the mean velocity from each component
* This focuses on velocity fluctuations rather than mean flow
* Essential for finding the dominant direction of variability

### Step 3: Calculate Covariance Matrix
C = (U.T @ U) / (len(U) - 1)

This creates a 2×2 covariance matrix:



C = [σ_EE   σ_EN]

       [σ_NE   σ_NN]

* σ_EE = variance of East component
* σ_NN = variance of North component
* σ_EN = σ_NE = covariance between East and North

### Step 4: Eigenvalue Decomposition
eigvals, eigvecs = np.linalg.eig(C)

* Eigenvalues: Represent variance in each principal direction
* Eigenvectors: Represent the principal directions themselves

### Step 6: Convert to Bearing
phi_deg = math.degrees(math.atan2(v[0], v[1])) % 360.0

* Converts the eigenvector to a compass bearing
* atan2(East_component, North_component) gives angle from North
* Result: bearing in degrees (0-360°, clockwise from North)

## 2. Physical Interpretation:

* Tidal flows are typically bidirectional along a channel
* The maximum variance occurs along the main flow axis
* Minimum variance occurs perpendicular to the channel

This technique automatically discovers the main flow direction from the data itself, without needing prior knowledge of the channel orientation.



## Transformed velocity components
After identifying the dominant flow direction using principal component analysis of the velocity data and finding the direction along which the velocity variance is maximum - typically aligned with the channel, we can find the velocity components after coordinate transformation from the original East-North coordinate system to a channel-aligned coordinate system:

### Original Velocity Components:
* Eas (East): Velocity component pointing East (positive eastward)
* Nor (North): Velocity component pointing North (positive northward)
### Transformed Velocity Components:
* u_along: Velocity component along the main channel axis
* u_across: Velocity component across the channel (perpendicular to main flow)

u_along  = Eas*np.sin(phi) + Nor*np.cos(phi)

u_across = Eas*np.cos(phi) - Nor*np.sin(phi)

### Coordinate Rotation:
The East-North velocities are rotated to align with this principal axis using the following function:

rotate_along_across(Eas, Nor, bearing_deg)

### Physical Meaning:
u_along (Along-Channel Velocity):
* Positive values: Flow in the flood direction (toward land/up-channel)
* Negative values: Flow in the ebb direction (toward sea/down-channel)
* Near zero: Slack water (minimal flow)

u_across (Cross-Channel Velocity):
* Positive values: Flow toward one side of the channel
* Negative values: Flow toward the other side of the channel
* Usually much smaller than u_along in well-defined channels

### Physical Interpretation:
* u_along represents the main tidal flow (what we care about for tidal analysis)
* u_across represents secondary circulation or measurement noise

### Long Island Sound ADCP data (station B):
* Bearing used: 255°T (flood direction toward land)
* Ebb direction: 75°T (toward ocean)
* When u_along > 0.05 m/s → FLOOD
* When u_along < -0.05 m/s → EBB
* When |u_along| < 0.05 m/s → SLACK

### Implementing the method:
The script "ebbFloodClassifier.py", we can find the principal axis by running:

python ebbFloodClassifier.py depthAvg_ADCPdata.csv --time-col "Date & Time.2" --east-col "Eas" --north-col "Nor" --estimate-axis

This gives us the values of 75.3°T for Principal axis estimate, and 255.3°T (75.3° + 180°) for the flood direction, based on geographical knowledge from Long Island Sound, in which the flood tides flow FROM the ocean INTO the Sound (westward, toward land), and the ebb tides flow FROM the Sound TO the ocean (eastward, toward sea). This matches very closely with the manually specified 75°.

### How to run the script:

python ebbFloodClassifier.py [CSV_FILE] [OPTIONS]

python ebbFloodClassifier.py --help

1. If you know the ebb direction (most common):
* python ebbFloodClassifier.py depthAvg_ADCPdata.csv --time-col "Date & Time.2" --east-col "Eas" --north-col "Nor" --ebbward-bearing 75 --plot

* python ebbFloodClassifier.py depthAvg_ADCPdata.csv --time-col "Date & Time.2" --east-col "Eas" --north-col "Nor" --floodward-bearing 255 --plot

* python ebbFloodClassifier.py depthAvg_ADCPdata.csv --time-col "Date & Time.2" --east-col "Eas" --north-col "Nor" --estimate-axis --plot

2. Additional flags:
* --out-csv "results/tidal_analysis.csv" : Output *.csv file.
* --thr 0.02 : Slack threshhold
* --smooth-n 5 : Smoothing windoe

### Output:
The "ebbFloodClassifier.py" will output a csv file, "depthAvg_ADCPdata_labeled.csv", in which the along and across velocities are calculated, and labeled by ebb and flood.